# N02 — Điền NULL cột age
**Võ Minh Huy — 2311110117**

Chỉ thực hiện phần tuổi. Notebook chạy độc lập bằng **Run All**: hai ô đầu dùng lại đúng code chuẩn bị và N01 đã hoàn thành, để tạo trạng thái đầu vào giống SQL. Không cần mở lại notebook sex. CSV before được giữ nguyên.

## 1. Chuẩn bị lại trạng thái sau N01
Phần này dùng lại code cũ; chụp minh chứng N02 ở mục 2 và mục 3 phía dưới.

In [1]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
from collections import Counter
import csv
import pandas as pd

thu_muc = Path.cwd()
if thu_muc.name == "python":
    thu_muc = thu_muc.parent
file_csv = thu_muc / "VoMinhHuy_Heart Failure Prediction_before.csv"
with file_csv.open(encoding="utf-8-sig", newline="") as f:
    du_lieu_goc = list(csv.DictReader(f))

def doc_so(gia_tri, cot):
    chuoi = gia_tri.strip()
    if cot == "sex":
        chuoi = {"m":"1", "male":"1", "f":"0", "female":"0"}.get(chuoi.lower(), chuoi)
    try:
        return Decimal(chuoi).quantize(Decimal("0.000001"))
    except InvalidOperation:
        return None

ho_so, da_gap = [], set()
for dong in du_lieu_goc:
    if not any(v.strip() for v in dong.values()):
        continue
    ban_ghi = {cot: doc_so(v, cot) for cot, v in dong.items()}
    khoa = tuple(ban_ghi.values())
    if khoa not in da_gap:
        da_gap.add(khoa)
        ho_so.append(ban_ghi)
df = pd.DataFrame(ho_so)
df.insert(0, "MaHoSo", range(1, len(df)+1))
mau_cho = df.loc[df["sex"].isin([0, 1])].copy()
pd.DataFrame({"SoHoSo":[len(df)], "SoSexThieuTruocDien":[df["sex"].isna().sum()]})

,SoHoSo,SoSexThieuTruocDien
0,299,20


In [2]:
cac_nhom = [
    ["anaemia", "diabetes", "smoking", "high_blood_pressure"],
    ["anaemia", "diabetes"], ["diabetes"], []
]
nhat_ky = []
for i in df.index[df["sex"].isna()]:
    ket_qua = {"MaHoSo":df.at[i, "MaHoSo"],
               **{c:df.at[i,c] for c in cac_nhom[0]},
               "CapNhom":None, "SoMauCho":0, "GiaTriDien":None}
    for cap, cac_cot in enumerate(cac_nhom, start=1):
        nhom = mau_cho
        for cot in cac_cot:
            nhom = nhom.loc[nhom[cot].eq(df.at[i,cot])]
        if len(nhom) < 5:
            continue
        tan_suat = Counter(nhom["sex"])
        mode = min(v for v, n in tan_suat.items() if n == max(tan_suat.values()))
        df.at[i, "sex"] = mode
        ket_qua.update(CapNhom=cap, SoMauCho=len(nhom), GiaTriDien=mode)
        break
    nhat_ky.append(ket_qua)
nhat_ky = pd.DataFrame(nhat_ky)
tong_ket = pd.DataFrame({"SoHoSo":[len(df)],
    "SoSexThieuTruocDien":[len(nhat_ky)],
    "SoODaDien":[nhat_ky["GiaTriDien"].notna().sum()],
    "SoSexThieuSauDien":[df["sex"].isna().sum()]})
tong_ket

,SoHoSo,SoSexThieuTruocDien,SoODaDien,SoSexThieuSauDien
0,299,20,20,0


## 2. Chuẩn bị tuổi và mẫu cho cố định
21 ô NULL/sai kiểu và 2 tuổi bị làm nhiễu (−12, 250) là 23 ô cần điền. Mẫu cho chỉ dùng tuổi quan sát hợp lệ. Khóa sex của mẫu cho lấy từ trước N01; hồ sơ nhận được dùng sex đã điền. Hồ sơ thiếu sex quan sát vẫn có thể cho tuổi ở cấp toàn cột.

In [3]:
from decimal import ROUND_HALF_UP
from statistics import median

# df là kết quả sau N01; ho_so giữ dữ liệu quan sát trước N01.
age_work = df.copy(deep=True)
truoc_n01 = pd.DataFrame(ho_so)
tuoi_nhieu = [Decimal("-12"), Decimal("250")]
mat_na_can_dien = age_work["age"].isna() | age_work["age"].isin(tuoi_nhieu)
age_targets = age_work.loc[mat_na_can_dien, ["MaHoSo", "age", "sex", "diabetes"]].copy()
age_donors = truoc_n01.loc[
    truoc_n01["age"].notna() & ~truoc_n01["age"].isin(tuoi_nhieu),
    ["age", "sex", "diabetes"]
].copy()
age_work.loc[age_work["age"].isin(tuoi_nhieu), "age"] = None

pd.DataFrame({"SoHoSo":[len(age_work)],
    "SoNullHoacSaiKieu":[age_targets["age"].isna().sum()],
    "SoTuoiNhieu":[age_targets["age"].isin(tuoi_nhieu).sum()],
    "SoOCanDien":[len(age_targets)]})

,SoHoSo,SoNullHoacSaiKieu,SoTuoiNhieu,SoOCanDien
0,299,21,2,23


## 3. Điền median theo nhóm và tổng kết
Cấp 1: cùng sex và diabetes → cấp 2: cùng sex → cấp 3: toàn cột. Dùng nhóm đầu có ít nhất 5 mẫu; thiếu mẫu ở mọi cấp thì giữ NULL. Chỉ giá trị vừa điền được làm tròn 1 chữ số theo ROUND_HALF_UP, tương ứng ROUND trong SQL.

In [4]:
cac_nhom_age = [["sex", "diabetes"], ["sex"], []]
dong_nhat_ky_age = []
for i, r in age_targets.iterrows():
    ket_qua_age = {"MaHoSo":r["MaHoSo"], "TuoiTruoc":r["age"],
        "sex":r["sex"], "diabetes":r["diabetes"],
        "LyDo":"Tuổi bị làm nhiễu" if r["age"] in tuoi_nhieu else "NULL hoặc sai kiểu",
        "CapNhom":None, "SoMauCho":0, "MedianNhom":None, "TuoiDien":None}
    for cap, cac_cot in enumerate(cac_nhom_age, start=1):
        nhom = age_donors
        for cot in cac_cot:
            nhom = nhom.loc[nhom[cot].eq(r[cot])]
        if len(nhom) < 5:
            continue
        median_nhom = median(nhom["age"].tolist())
        tuoi_dien = median_nhom.quantize(Decimal("0.1"), rounding=ROUND_HALF_UP)
        age_work.at[i, "age"] = tuoi_dien
        ket_qua_age.update(CapNhom=cap, SoMauCho=len(nhom),
                          MedianNhom=median_nhom, TuoiDien=tuoi_dien)
        break
    dong_nhat_ky_age.append(ket_qua_age)
nhat_ky_age = pd.DataFrame(dong_nhat_ky_age)
tong_ket_age = pd.DataFrame({"SoHoSo":[len(age_work)],
    "SoNullHoacSaiKieu":[age_targets["age"].isna().sum()],
    "SoTuoiNhieu":[age_targets["age"].isin(tuoi_nhieu).sum()],
    "SoOCanDien":[len(age_targets)],
    "SoODaDien":[nhat_ky_age["TuoiDien"].notna().sum()],
    "SoAgeThieuSauDien":[age_work["age"].isna().sum()]})
tong_ket_age

,SoHoSo,SoNullHoacSaiKieu,SoTuoiNhieu,SoOCanDien,SoODaDien,SoAgeThieuSauDien
0,299,21,2,23,23,0


## 4. Nhật ký 23 ô tuổi
Chụp cả bảng để thấy tuổi trước xử lý, lý do, cấp nhóm, số mẫu cho, median và tuổi điền. MaHoSo là số thứ tự nội bộ, có thể khác SQL.

In [5]:
nhat_ky_age

,MaHoSo,TuoiTruoc,sex,diabetes,LyDo,CapNhom,SoMauCho,MedianNhom,TuoiDien
0,6,None,1.000000,0.000000,NULL hoặc sai kiểu,1,103,60.000000,60.0
1,10,None,1.000000,0.000000,NULL hoặc sai kiểu,1,103,60.000000,60.0
2,11,250.000000,1.000000,0.000000,Tuổi bị làm nhiễu,1,103,60.000000,60.0
3,26,-12.000000,1.000000,1.000000,Tuổi bị làm nhiễu,1,58,60.833500,60.8
4,46,None,1.000000,1.000000,NULL hoặc sai kiểu,1,58,60.833500,60.8
5,58,None,1.000000,0.000000,NULL hoặc sai kiểu,1,103,60.000000,60.0
6,61,None,1.000000,1.000000,NULL hoặc sai kiểu,1,58,60.833500,60.8
7,64,None,1.000000,0.000000,NULL hoặc sai kiểu,1,103,60.000000,60.0
8,78,None,1.000000,1.000000,NULL hoặc sai kiểu,1,58,60.833500,60.8
9,94,None,1.000000,0.000000,NULL hoặc sai kiểu,1,103,60.000000,60.0


## 5. Lưu kết quả của N02
Xuất bảng tổng kết, nhật ký và bản làm việc trung gian. Chưa xử lý các cột còn lại, chưa tạo after cuối.

In [6]:
thu_muc_age = thu_muc / "ket_qua" / "06_dien_age"
thu_muc_age.mkdir(parents=True, exist_ok=True)
tong_ket_age.to_csv(thu_muc_age / "tong_ket_python.csv", index=False, encoding="utf-8-sig")
nhat_ky_age.to_csv(thu_muc_age / "nhat_ky_python.csv", index=False, encoding="utf-8-sig")
age_work.to_csv(thu_muc_age / "ban_lam_viec_sau_age_python.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"TrangThai":["Đã lưu kết quả riêng N02; CSV before được giữ nguyên."]})

,TrangThai
0,Đã lưu kết quả riêng N02; CSV before được giữ ...
